# Bronze HTML Profile Parser with PySpark

This notebook uses PySpark to read every HTML file from the bronze layer, extract structured profile fields, and consolidate the result into a Spark DataFrame named `df_bronze`.

In [12]:
from datetime import datetime
from html import unescape
import json
from pathlib import Path
import re
from urllib.parse import parse_qs, unquote, urlparse

from pyspark.sql import SparkSession, functions as F, types as T

In [2]:
# Resolve the project root whether the notebook is launched from the repository root or from the notebooks folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Bronze is the raw ingestion layer that stores the source HTML files.
BRONZE_DIR = PROJECT_ROOT / "data" / "bronze"


def get_spark_session(app_name="Leadforge Bronze Parser"):
    """Create or reuse the Spark session used by this notebook.

    The notebook runs locally, so this helper creates a local Spark session
    when needed and reuses it on subsequent calls. The log level is lowered
    to keep the notebook output focused on data exploration instead of Spark
    internals.
    """
    spark_session = SparkSession.builder.master("local[*]").appName(app_name).getOrCreate()
    spark_session.sparkContext.setLogLevel("WARN")
    return spark_session


spark = get_spark_session()

spark

In [16]:
def extract_rating_and_review_count(html_text):
    """Extract overall rating and review count from JSON-LD schema.

    Searches for <script type="application/ld+json"> blocks in the HTML,
    parses them as JSON, and recursively searches for aggregateRating objects.

    Parameters
    ----------
    html_text : str
        Raw HTML content of the profile page.

    Returns
    -------
    tuple of (float or None, int or None)
        The overall rating value and review count, or (None, None) if not found.
    """
    if not isinstance(html_text, str) or not html_text.strip():
        return None, None

    # Find all JSON-LD script blocks
    script_pattern = re.compile(
        r'<script[^>]*type\s*=\s*["\']application/ld\+json["\'][^>]*>(?P<content>.*?)</script>',
        flags=re.IGNORECASE | re.DOTALL,
    )

    def find_aggregate_rating(obj):
        """Recursively search for aggregateRating in a nested dict/list."""
        if isinstance(obj, dict):
            # Check if this dict is an aggregateRating or contains both fields
            if obj.get("@type") == "AggregateRating" or ("ratingValue" in obj and "reviewCount" in obj):
                rating_value = obj.get("ratingValue")
                review_count = obj.get("reviewCount")
                if rating_value is not None and review_count is not None:
                    try:
                        return float(rating_value), int(review_count)
                    except (ValueError, TypeError):
                        pass

            # Recursively search in all dict values
            for value in obj.values():
                result = find_aggregate_rating(value)
                if result != (None, None):
                    return result

        elif isinstance(obj, list):
            # Recursively search in list items
            for item in obj:
                result = find_aggregate_rating(item)
                if result != (None, None):
                    return result

        return None, None

    # Process each JSON-LD script block
    for script_match in script_pattern.finditer(html_text):
        json_str = script_match.group("content")
        try:
            data = json.loads(json_str)
            result = find_aggregate_rating(data)
            if result != (None, None):
                return result
        except (json.JSONDecodeError, ValueError, TypeError):
            # Skip invalid JSON blocks
            continue

    return None, None

In [4]:
def load_bronze_html_files(spark, bronze_dir=BRONZE_DIR):
    """Load every HTML file found under the bronze directory.

    The binary file source returns one row per file, which is a good fit for
    raw HTML because the full document can be preserved as a single string.
    Metadata is attached so the HTML rows follow a consistent shape for the
    cleaning step that will happen next.
    """
    frames = []

    for file_path in sorted(bronze_dir.rglob("*.html")):
        metadata = build_file_metadata(file_path, bronze_dir)
        html_df = spark.read.format("binaryFile").load(str(file_path)).select(
            F.lit("html").alias("record_type"),
            F.lit(metadata["source_file"]).alias("source_file"),
            F.lit(metadata["source_folder"]).alias("source_folder"),
            F.lit(metadata["file_name"]).alias("file_name"),
            F.lit(metadata["extension"]).alias("extension"),
            F.col("length").cast("long").alias("file_size_bytes"),
            F.lit(metadata.get("company_slug")).alias("company_slug"),
            F.lit(metadata.get("collected_at")).alias("collected_at"),
            F.lit(metadata.get("file_sequence")).cast("int").alias("file_sequence"),
            F.expr("decode(content, 'UTF-8')").alias("html"),
        )
        frames.append(html_df)

    if not frames:
        return spark.createDataFrame([], "record_type STRING")

    combined_df = frames[0]
    for frame in frames[1:]:
        combined_df = combined_df.unionByName(frame, allowMissingColumns=True)

    return combined_df



def load_bronze_dataframe(spark, bronze_dir=BRONZE_DIR):
    """Build the bronze DataFrame from HTML sources only.

    The function validates that the bronze directory exists, loads the HTML
    subset, and reorders the columns so the most useful fields appear first in
    notebook previews.
    """
    if not bronze_dir.exists():
        raise FileNotFoundError(f"Bronze directory not found: {bronze_dir}")

    if not list(bronze_dir.rglob("*.html")):
        return spark.createDataFrame([], "record_type STRING")

    df = load_bronze_html_files(spark, bronze_dir)

    # Keep the most relevant columns at the front for notebook inspection.
    first_columns = [
        "record_type",
        "company_slug",
        "source_folder",
        "source_file",
        "file_name",
        "extension",
        "file_size_bytes",
        "file_sequence",
        "collected_at",
        "html",
    ]
    ordered_columns = [column for column in first_columns if column in df.columns]
    ordered_columns.extend(column for column in df.columns if column not in ordered_columns)

    return df.select(*ordered_columns)

In [5]:
df_bronze = load_bronze_dataframe(spark)

df_bronze

DataFrame[record_type: string, company_slug: string, source_folder: string, source_file: string, file_name: string, extension: string, file_size_bytes: bigint, file_sequence: int, collected_at: timestamp, html: string]

In [17]:
profile_summary_schema = T.StructType([
    T.StructField("minimum_project_size", T.StringType(), True),
    T.StructField("hourly_rate", T.StringType(), True),
    T.StructField("employee_range", T.StringType(), True),
    T.StructField("locations", T.StringType(), True),
    T.StructField("year_founded", T.IntegerType(), True),
    T.StructField("languages", T.StringType(), True),
])

rating_and_count_schema = T.StructType([
    T.StructField("overall_rating", T.DoubleType(), True),
    T.StructField("review_count", T.IntegerType(), True),
])


def rating_and_count_udf_func(html_text):
    """Wrapper to return a row for the schema."""
    overall_rating, review_count = extract_rating_and_review_count(html_text)
    return (overall_rating, review_count)


profile_title_udf = F.udf(extract_profile_title, T.StringType())
website_url_udf = F.udf(extract_website_url, T.StringType())
profile_description_udf = F.udf(extract_profile_description, T.StringType())
services_udf = F.udf(extract_services, T.StringType())
rating_and_count_udf = F.udf(rating_and_count_udf_func, rating_and_count_schema)
profile_summary_udf = F.udf(extract_profile_summary_details, profile_summary_schema)


def enrich_profile_dataframe(df):
    """Add extracted profile fields to the raw HTML dataframe."""
    enriched_df = (
        df.withColumn("profile_title", profile_title_udf(F.col("html")))
        .withColumn("website_url", website_url_udf(F.col("html")))
        .withColumn("profile_description", profile_description_udf(F.col("html")))
        .withColumn("services", services_udf(F.col("html")))
        .withColumn("rating_and_count", rating_and_count_udf(F.col("html")))
        .withColumn("overall_rating", F.col("rating_and_count.overall_rating"))
        .withColumn("review_count", F.col("rating_and_count.review_count"))
        .withColumn("profile_summary", profile_summary_udf(F.col("html")))
        .withColumn("minimum_project_size", F.col("profile_summary.minimum_project_size"))
        .withColumn("hourly_rate", F.col("profile_summary.hourly_rate"))
        .withColumn("employee_range", F.col("profile_summary.employee_range"))
        .withColumn("locations", F.col("profile_summary.locations"))
        .withColumn("year_founded", F.col("profile_summary.year_founded"))
        .withColumn("languages", F.col("profile_summary.languages"))
        .drop("profile_summary", "rating_and_count")
    )

    return enriched_df


df_bronze = enrich_profile_dataframe(df_bronze)

df_bronze

DataFrame[record_type: string, company_slug: string, source_folder: string, source_file: string, file_name: string, extension: string, file_size_bytes: bigint, file_sequence: int, collected_at: timestamp, html: string, profile_title: string, website_url: string, minimum_project_size: string, hourly_rate: string, employee_range: string, locations: string, year_founded: int, languages: string, profile_description: string, services: string, overall_rating: double, review_count: int]

In [18]:
# Show a small vertical preview so the extracted fields are easy to read in the notebook.
preview_columns = [
    column
    for column in [
        "company_slug",
        "profile_title",
        "website_url",
        "overall_rating",
        "review_count",
        "services",
        "profile_description",
        "minimum_project_size",
        "hourly_rate",
        "employee_range",
        "locations",
        "year_founded",
        "languages",
        "source_folder",
        "file_name",
    ]
    if column in df_bronze.columns
]

preview_df = df_bronze.select(*preview_columns).limit(5)
preview_df.show(vertical=True, truncate=False)

-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------